# START HERE — verify the cheap model before spending more

**Run this notebook, not v2 or v3, for the next test.**

1. Tests **GPT-5 nano** first, with **one group per condition**, four roles per group, and 36 weeks: **16 agents / 576 game decisions**.
2. Makes **4 ordering checks + 16 rules checks + 576 decisions = 596 planned requests**. This is a full-game measurement pilot, not another isolated-state screen.
3. Uses a **$0.75 local spending allowance**. A same-token estimate at 2,000 input + 1,000–2,000 output tokens per request is roughly **$0.30–$0.54**, but the failed trial already showed that high effort can exceed 8,192 tokens on one decision. That earlier planning range is not a reliable forecast for this model. Actual spending can be higher and the guard may stop the pilot. The allowance is not your account balance or a guaranteed invoice cap.
4. Restores standard cost scoring: **0.50 holding + 1.00 backlog**, no factory charge and no sales revenue. Initial stock is 12, initial flow 4; customer demand is 4 for four weeks, then 8. Agents choose orders from week 1.
5. The final report tells us whether replies are valid, β is measurable, low/high effort differs, and the run is affordable. **It does not test H1/H2 statistically** with only one group per condition.
6. It never switches models or launches a larger experiment automatically. GPT-5.4 nano would require a separate small verification after we review these results.

**Speed update:** up to **8 independent requests at once**, across groups and roles. Weeks within each group still run in order. The stopped startup sample took about 6 seconds per low-effort order and 50 seconds per high-effort order. A rough full-game planning range is **40–90 minutes**, not a few minutes; later states and rate limits can make it longer. The old sequential version could take 4–5 hours. Parallelism does not reduce token cost. Four startup responses used enough output tokens to project roughly $0.84 in game output alone if that average persisted; the $0.75 guard may stop this run. This four-response estimate is uncertain.

**Instructions:** keep this notebook beside `budget_verification.py`. Restart kernel → Run All → enter your API key in the hidden popup. When finished or stopped, run the final report cell if necessary, then tell me **“the verification finished”**. I can read the saved files directly. Do not paste your key into chat or a code cell.

For a free software check instead, set MODE = "offline" before Run All. Offline results do not verify real model access or behavior.

**Correction after the first stopped trial:** GPT-5 nano exhausted 8,192 tokens before producing its first order. This fresh pilot gives both conditions 25,000 tokens per response and a common 300-second infrastructure timeout, while retaining the $0.75 total local allowance. It preserves the earlier failed run and does not retry it or change reasoning effort. A higher ceiling is not a promise of completion or affordability.


In [ ]:
# Uncomment if dependencies are missing, then restart the kernel.
# %pip install "openai>=2,<3" "pydantic>=2.8,<3" numpy pandas scipy matplotlib ipykernel
from pathlib import Path
import sys, importlib, os, getpass
from IPython.display import display, Markdown, Image
BASE = Path.cwd()
if (BASE / "Project" / "beer_game_experiment").is_dir():
    BASE = BASE / "Project" / "beer_game_experiment"
elif (BASE / "beer_game_experiment").is_dir():
    BASE = BASE / "beer_game_experiment"
if not (BASE / "budget_verification.py").exists():
    raise FileNotFoundError("Open this notebook beside budget_verification.py in beer_game_experiment.")
sys.path.insert(0, str(BASE))
import budget_verification as study
importlib.reload(study)


## 1. Settings — already prepared for the small test

Keep these settings for this verification. Both effort levels use the same model, instructions except training, 25,000-token ceiling and 300-second infrastructure timeout. Reasoning tokens count toward output usage. There is no 60/120-second human time-pressure claim.

OpenAI lists this model snapshot for shutdown on **December 11, 2026**. Deprecated does not mean already unavailable; the first four requests check real ordering responses in all conditions. If unavailable, the notebook stops without silently choosing another model. [Model pricing](https://developers.openai.com/api/docs/models/gpt-5-nano) · [Deprecation schedule](https://developers.openai.com/api/docs/deprecations)


In [ ]:
MODE = "api"                    # Use "offline" for a free software check
BUDGET_USD = 0.75                # Local allowance across this entire verification run
REPLICATE_ID = "cheap-model-parallel-03"
MAX_CONCURRENT_REQUESTS = 8      # Shared limit across all groups and roles
CONFIG = dict(mode=MODE, stage="game", model=study.MODEL,
              groups_per_cell=1, weeks=36, replicate_id=REPLICATE_ID,
              seed=20260927, budget_usd=BUDGET_USD,
              max_output_tokens=25000, timeout_s=300,
              max_concurrent_requests=MAX_CONCURRENT_REQUESTS)
print("Model:", study.MODEL)
print("Concurrent requests:", MAX_CONCURRENT_REQUESTS, "| planning estimate: 40–90 minutes, uncertain")
print("Mode:", MODE, "| 4 groups / 16 agents / 36 weeks / 596 planned responses")
print(f"Local allowance: ${BUDGET_USD:.2f}; no automatic model upgrade or increase in allowance.")
print("Keep settings unchanged to reuse successful cached requests. Only raising the allowance resumes a spending stop.")


## 2. Free software verification

Known-answer β recovery and a demand-copying counterexample check the analysis code before paid requests. Passing is not evidence that GPT will follow the assumed ordering model. We judge the real pilot by measurement quality, never by whether the training effect has the desired sign.


In [ ]:
checks = study.measurement_self_check()
display(checks)
assert checks.passed.all(), "Estimator self-check failed; do not proceed to paid requests."


## 3. Run the verification — this is the only paid cell

The first four requests check actual ordering decisions under control/trained × low/high reasoning with structured output. Then four full games run. All roles use the same weekly snapshot before orders are committed. Failed, refused, incomplete or incorrect replies stop the pilot; they are not replaced with invented orders. The progress bar includes access checks and rules checks.

If stopped, the next cell still reports what happened. Do not keep rerunning a failed request or spend the rest of your $3 without reviewing it. A spending stop can resume successful cached requests if you deliberately raise only BUDGET_USD; a saved failed response requires review.

Independent groups and same-week roles run concurrently, with at most 8 requests in flight. The bar shows completed/remaining requests, active requests, and each group's week. On failure, no new requests are sent; already-sent requests finish and save their usage before stopping. The total spending reservation includes every in-flight request. If interrupted, unknown usage stays reserved. Rate-limit errors stop the run without automatic retries.


In [ ]:
client = None
results = None
run_dir = study.latest_run(BASE, CONFIG)
try:
    if MODE == "api":
        from openai import AsyncOpenAI
        key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ")
        if not key.strip():
            raise ValueError("An API key is required in API mode.")
        client = AsyncOpenAI(api_key=key.strip(), max_retries=0, timeout=CONFIG["timeout_s"])
        del key
    results, run_dir = await study.run_study(BASE, CONFIG, client=client)
    print("VERIFICATION COMPLETE. Run the next cell for the results.")
except study.ExperimentStopped as exc:
    print(str(exc))
    print("Run the next cell to review the saved status. No automatic upgrade or retry.")
finally:
    if client is not None:
        await client.close()
    print("Results folder:", run_dir)


## 4. Results — read these before paying for another run

The summary shows:

- Completed vs stopped, successful replies and failures, estimated spending.
- Supply-chain cost, customer fulfillment, inventory/backlog/order graphs.
- Usable β for each of the 16 roles and each complete group; blanks mean unidentified.
- Zero-order rate and agents that copied demand every week. Zero orders alone are not failures.
- Actual low/high reasoning-token use.
- Cost scenarios for larger GPT-5 nano runs and **repricing the same token counts** at GPT-5.4 nano rates. Repricing is not a prediction of another model's actual token use.

A promising screen means complete validated games, measurable β in all four groups, and descriptive effort separation. It does not establish hypotheses or justify selecting whichever model produces a preferred result. If it fails these checks, keep the results and review the cause before scaling. A different model still needs its own small verification.


In [ ]:
if (run_dir / "progress.json").exists():
    report = study.verification_report(run_dir, CONFIG)
    display(Markdown(report))
    figure = run_dir / "supply_chain_performance.png"
    if figure.exists():
        display(Image(filename=str(figure)))
    print("Report:", run_dir / "VERIFICATION_REPORT.md")
    print("Summary:", run_dir / "verification_summary.json")
    print("Tell me: 'the verification finished'. I can inspect these saved files.")
else:
    print("No run records yet. Complete the run cell or report the access/setup error.")
